In [1]:
import xarray as xr
import numpy as np
import yaml

In [2]:
forecast_timeframe = slice("2020-02-12T01:00:00", "2020-02-12T04:00:00")

In [107]:
## Preparation of config files

In [3]:
for domain_name in [f'domain{str(i).zfill(2)}' for i in range(2,14)]:
    with open(f"../data/config.{domain_name}.yaml", "r") as f:
        config = yaml.safe_load(f)
        config_domain = config.copy()
        config_domain['datastore']['config_path'] = f"/home/has/repos/mllam-exps-ShCu/data/datastore.interior.{domain_name}.ar.yaml"
        config_domain['datastore_boundary']['config_path'] = f"/home/has/repos/mllam-exps-ShCu/data/datastore.boundary.{domain_name}.ar.yaml"
        with open(f"../data/config.{domain_name}.ar.yaml", "w") as f_ar:
            yaml.dump(config_domain, f_ar)

    with open(f"../data/datastore.interior.{domain_name}.yaml", "r") as f:
        datastore_interior = yaml.safe_load(f)
        datastore_interior_ar = datastore_interior.copy()
        datastore_interior_ar['output']['splitting']['splits'] = {
            'test': {
                'end': '2020-02-12T03:00',
                'start': '2020-02-12T00:00'
            },
        }
        datastore_interior_ar['output']['coord_ranges'] = {
            'time': {
                'end': '2020-02-12T03:00',
                'start': '2020-02-12T00:00'
            }
        }
    with open(f"../data/datastore.interior.{domain_name}.ar.yaml", "w") as f_ar:
        yaml.dump(datastore_interior_ar, f_ar)
    
    with open(f"../data/datastore.boundary.{domain_name}.yaml", "r") as f:
        datastore_boundary = yaml.safe_load(f)
        datastore_boundary_ar = datastore_boundary.copy()
        datastore_boundary_ar['output']['splitting']['splits'] = {
            'test': {
                'end': '2020-02-12T03:00',
                'start': '2020-02-12T00:00'
            },
        }
        datastore_boundary_ar['output']['coord_ranges'] = {
            'time': {
                'end': '2020-02-12T03:00',
                'start': '2020-02-12T00:00'
            }
        }
    with open(f"../data/datastore.boundary.{domain_name}.ar.yaml", "w") as f_ar:
        yaml.dump(datastore_boundary_ar, f_ar)

In [ ]:
# now the datastores need to be created with mllam-data-prep.
# e.g. uv run python -m mllam_data_prep data/datastore.interior.domain02.ar.yaml

## Interior datastore

In [76]:
ds = xr.open_zarr("../data/datastore.interior.domain03.zarr")
del ds.attrs['creation_config']

In [88]:
ds_forecast = xr.open_zarr("../example_loamy_raid")
ds_forecast

/tmp/ipykernel_498871/658849900.py:1: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than None. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  ds_forecast = xr.open_zarr("../example_loamy_raid")


<xarray.Dataset> Size: 49GB
Dimensions:                    (elapsed_forecast_duration: 12,
                                grid_index: 524288, start_time: 325,
                                state_feature: 6)
Coordinates:
  * elapsed_forecast_duration  (elapsed_forecast_duration) timedelta64[ns] 96B ...
  * grid_index                 (grid_index) int64 4MB 11255161 ... 3934314
  * start_time                 (start_time) datetime64[ns] 3kB 2020-02-12T00:...
  * state_feature              (state_feature) <U8 192B 't_2m' ... 'tqc_dia'
    time                       (elapsed_forecast_duration) datetime64[ns] 96B dask.array<chunksize=(1,), meta=np.ndarray>
Data variables:
    state                      (start_time, elapsed_forecast_duration, grid_index, state_feature) float32 49GB dask.array<chunksize=(1, 1, 524288, 6), meta=np.ndarray>

In [ ]:
assert np.all(ds_forecast.grid_index.values == ds.grid_index.values), "Grid indices do not match!"

In [89]:
ds_forecast_sel = ds_forecast.sel(start_time=forecast_timeframe.start).drop_vars('time').rename({'elapsed_forecast_duration':'time'})
ds_forecast_sel

<xarray.Dataset> Size: 155MB
Dimensions:        (time: 12, grid_index: 524288, state_feature: 6)
Coordinates:
  * time           (time) timedelta64[ns] 96B 00:10:00 00:20:00 ... 02:00:00
  * grid_index     (grid_index) int64 4MB 11255161 11255162 ... 3934313 3934314
    start_time     datetime64[ns] 8B 2020-02-12T01:00:00
  * state_feature  (state_feature) <U8 192B 't_2m' 'u_10m' ... 'qv_2m' 'tqc_dia'
Data variables:
    state          (time, grid_index, state_feature) float32 151MB dask.array<chunksize=(1, 524288, 6), meta=np.ndarray>

In [90]:
ds_forecast_sel['time'] = ds_forecast_sel.start_time + ds_forecast_sel.time

In [91]:
ds_forecast_sel

<xarray.Dataset> Size: 155MB
Dimensions:        (time: 12, grid_index: 524288, state_feature: 6)
Coordinates:
  * time           (time) datetime64[ns] 96B 2020-02-12T01:10:00 ... 2020-02-...
  * grid_index     (grid_index) int64 4MB 11255161 11255162 ... 3934313 3934314
    start_time     datetime64[ns] 8B 2020-02-12T01:00:00
  * state_feature  (state_feature) <U8 192B 't_2m' 'u_10m' ... 'qv_2m' 'tqc_dia'
Data variables:
    state          (time, grid_index, state_feature) float32 151MB dask.array<chunksize=(1, 524288, 6), meta=np.ndarray>

In [92]:
ds_sel = ds.sel(time=forecast_timeframe)
ds_sel

<xarray.Dataset> Size: 495MB
Dimensions:                         (grid_index: 524288, forcing_feature: 3,
                                     time: 19, split_name: 3, split_part: 2,
                                     state_feature: 6, static_feature: 1)
Coordinates: (12/18)
    clat                            (grid_index) float64 4MB dask.array<chunksize=(524288,), meta=np.ndarray>
    clon                            (grid_index) float64 4MB dask.array<chunksize=(524288,), meta=np.ndarray>
  * forcing_feature                 (forcing_feature) <U15 180B 'hour_of_day_...
    forcing_feature_long_name       (forcing_feature) <U47 564B dask.array<chunksize=(3,), meta=np.ndarray>
    forcing_feature_source_dataset  (forcing_feature) <U16 192B dask.array<chunksize=(3,), meta=np.ndarray>
    forcing_feature_units           (forcing_feature) <U7 84B dask.array<chunksize=(3,), meta=np.ndarray>
    ...                              ...
    state_feature_units             (state_feature) <U8 192B dask.array<chunksize=(6,), meta=np.ndarray>
  * static_feature                  (static_feature) <U3 12B 'lsm'
    static_feature_long_name        (static_feature) <U1 4B dask.array<chunksize=(1,), meta=np.ndarray>
    static_feature_source_dataset   (static_feature) <U15 60B dask.array<chunksize=(1,), meta=np.ndarray>
    static_feature_units            (static_feature) <U1 4B dask.array<chunksize=(1,), meta=np.ndarray>
  * time                            (time) datetime64[ns] 152B 2020-02-12T01:...
Data variables: (12/14)
    forcing                         (forcing_feature, time, grid_index) float64 239MB dask.array<chunksize=(3, 1, 524288), meta=np.ndarray>
    forcing__train__diff_mean       (forcing_feature) float64 24B dask.array<chunksize=(3,), meta=np.ndarray>
    forcing__train__diff_std        (forcing_feature) float64 24B dask.array<chunksize=(3,), meta=np.ndarray>
    forcing__train__mean            (forcing_feature) float64 24B dask.array<chunksize=(3,), meta=np.ndarray>
    forcing__train__std             (forcing_feature) float64 24B dask.array<chunksize=(3,), meta=np.ndarray>
    splits                          (split_name, split_part) <U16 384B dask.array<chunksize=(3, 2), meta=np.ndarray>
    ...                              ...
    state__train__diff_std          (state_feature) float32 24B dask.array<chunksize=(6,), meta=np.ndarray>
    state__train__mean              (state_feature) float32 24B dask.array<chunksize=(6,), meta=np.ndarray>
    state__train__std               (state_feature) float32 24B dask.array<chunksize=(6,), meta=np.ndarray>
    static                          (static_feature, grid_index) float64 4MB dask.array<chunksize=(1, 524288), meta=np.ndarray>
    static__train__mean             (static_feature) float64 8B dask.array<chunksize=(1,), meta=np.ndarray>
    static__train__std              (static_feature) float64 8B dask.array<chunksize=(1,), meta=np.ndarray>
Attributes:
    created_on:       2025-10-11T11:01:47
    created_with:     mllam-data-prep (https://github.com/mllam/mllam-data-prep)
    dataset_version:  v1.0.0
    mdp_version:      v0.5.0
    schema_version:   v0.6.0

In [94]:
ds_merged = xr.merge([ds_forecast_sel, ds_sel],compat='override')

In [95]:
ds_sel.state.sel(time="2020-02-12T01:20:00", state_feature='u_10m').min().compute()

<xarray.DataArray 'state' ()> Size: 4B
array(-17.17142, dtype=float32)
Coordinates:
    state_feature                 <U8 32B 'u_10m'
    state_feature_long_name       <U17 68B 'zonal wind in 10m'
    state_feature_source_dataset  <U11 44B 'icon_merged'
    state_feature_units           <U5 20B 'm s-1'
    time                          datetime64[ns] 8B 2020-02-12T01:20:00

In [96]:
ds_forecast_sel.state.sel(time="2020-02-12T01:20:00", state_feature='u_10m').min().compute()

<xarray.DataArray 'state' ()> Size: 4B
array(-13.671075, dtype=float32)
Coordinates:
    time           datetime64[ns] 8B 2020-02-12T01:20:00
    start_time     datetime64[ns] 8B 2020-02-12T01:00:00
    state_feature  <U8 32B 'u_10m'

In [97]:
ds_merged.state.sel(time="2020-02-12T01:20:00", state_feature='u_10m').min().compute()

<xarray.DataArray 'state' ()> Size: 4B
array(-13.671075, dtype=float32)
Coordinates:
    time                          datetime64[ns] 8B 2020-02-12T01:20:00
    state_feature                 <U8 32B 'u_10m'
    start_time                    datetime64[ns] 8B 2020-02-12T01:00:00
    state_feature_long_name       <U17 68B 'zonal wind in 10m'
    state_feature_source_dataset  <U11 44B 'icon_merged'
    state_feature_units           <U5 20B 'm s-1'

In [ ]:
ds_merged # modified interior

<xarray.Dataset> Size: 495MB
Dimensions:                         (time: 19, grid_index: 524288,
                                     state_feature: 6, forcing_feature: 3,
                                     split_name: 3, split_part: 2,
                                     static_feature: 1)
Coordinates: (12/19)
  * time                            (time) datetime64[ns] 152B 2020-02-12T01:...
  * grid_index                      (grid_index) int64 4MB 11255161 ... 3934314
  * state_feature                   (state_feature) <U8 192B 't_2m' ... 'tqc_...
    start_time                      datetime64[ns] 8B 2020-02-12T01:00:00
  * forcing_feature                 (forcing_feature) <U15 180B 'hour_of_day_...
  * split_name                      (split_name) <U5 60B 'test' 'train' 'val'
    ...                              ...
    state_feature_long_name         (state_feature) <U48 1kB dask.array<chunksize=(6,), meta=np.ndarray>
    state_feature_source_dataset    (state_feature) <U11 264B dask.array<chunksize=(6,), meta=np.ndarray>
    state_feature_units             (state_feature) <U8 192B dask.array<chunksize=(6,), meta=np.ndarray>
    static_feature_long_name        (static_feature) <U1 4B dask.array<chunksize=(1,), meta=np.ndarray>
    static_feature_source_dataset   (static_feature) <U15 60B dask.array<chunksize=(1,), meta=np.ndarray>
    static_feature_units            (static_feature) <U1 4B dask.array<chunksize=(1,), meta=np.ndarray>
Data variables: (12/14)
    state                           (time, grid_index, state_feature) float32 239MB dask.array<chunksize=(1, 524288, 6), meta=np.ndarray>
    forcing                         (forcing_feature, time, grid_index) float64 239MB dask.array<chunksize=(3, 1, 524288), meta=np.ndarray>
    forcing__train__diff_mean       (forcing_feature) float64 24B dask.array<chunksize=(3,), meta=np.ndarray>
    forcing__train__diff_std        (forcing_feature) float64 24B dask.array<chunksize=(3,), meta=np.ndarray>
    forcing__train__mean            (forcing_feature) float64 24B dask.array<chunksize=(3,), meta=np.ndarray>
    forcing__train__std             (forcing_feature) float64 24B dask.array<chunksize=(3,), meta=np.ndarray>
    ...                              ...
    state__train__diff_std          (state_feature) float32 24B dask.array<chunksize=(6,), meta=np.ndarray>
    state__train__mean              (state_feature) float32 24B dask.array<chunksize=(6,), meta=np.ndarray>
    state__train__std               (state_feature) float32 24B dask.array<chunksize=(6,), meta=np.ndarray>
    static                          (static_feature, grid_index) float64 4MB dask.array<chunksize=(1, 524288), meta=np.ndarray>
    static__train__mean             (static_feature) float64 8B dask.array<chunksize=(1,), meta=np.ndarray>
    static__train__std              (static_feature) float64 8B dask.array<chunksize=(1,), meta=np.ndarray>

In [75]:
ds_sel

<xarray.Dataset> Size: 495MB
Dimensions:                         (grid_index: 524288, forcing_feature: 3,
                                     time: 19, split_name: 3, split_part: 2,
                                     state_feature: 6, static_feature: 1)
Coordinates: (12/18)
    clat                            (grid_index) float64 4MB dask.array<chunksize=(524288,), meta=np.ndarray>
    clon                            (grid_index) float64 4MB dask.array<chunksize=(524288,), meta=np.ndarray>
  * forcing_feature                 (forcing_feature) <U15 180B 'hour_of_day_...
    forcing_feature_long_name       (forcing_feature) <U47 564B dask.array<chunksize=(3,), meta=np.ndarray>
    forcing_feature_source_dataset  (forcing_feature) <U16 192B dask.array<chunksize=(3,), meta=np.ndarray>
    forcing_feature_units           (forcing_feature) <U7 84B dask.array<chunksize=(3,), meta=np.ndarray>
    ...                              ...
    state_feature_units             (state_feature) <U8 192B dask.array<chunksize=(6,), meta=np.ndarray>
  * static_feature                  (static_feature) <U3 12B 'lsm'
    static_feature_long_name        (static_feature) <U1 4B dask.array<chunksize=(1,), meta=np.ndarray>
    static_feature_source_dataset   (static_feature) <U15 60B dask.array<chunksize=(1,), meta=np.ndarray>
    static_feature_units            (static_feature) <U1 4B dask.array<chunksize=(1,), meta=np.ndarray>
  * time                            (time) datetime64[ns] 152B 2020-02-12T01:...
Data variables: (12/14)
    forcing                         (forcing_feature, time, grid_index) float64 239MB dask.array<chunksize=(3, 1, 524288), meta=np.ndarray>
    forcing__train__diff_mean       (forcing_feature) float64 24B dask.array<chunksize=(3,), meta=np.ndarray>
    forcing__train__diff_std        (forcing_feature) float64 24B dask.array<chunksize=(3,), meta=np.ndarray>
    forcing__train__mean            (forcing_feature) float64 24B dask.array<chunksize=(3,), meta=np.ndarray>
    forcing__train__std             (forcing_feature) float64 24B dask.array<chunksize=(3,), meta=np.ndarray>
    splits                          (split_name, split_part) <U16 384B dask.array<chunksize=(3, 2), meta=np.ndarray>
    ...                              ...
    state__train__diff_std          (state_feature) float32 24B dask.array<chunksize=(6,), meta=np.ndarray>
    state__train__mean              (state_feature) float32 24B dask.array<chunksize=(6,), meta=np.ndarray>
    state__train__std               (state_feature) float32 24B dask.array<chunksize=(6,), meta=np.ndarray>
    static                          (static_feature, grid_index) float64 4MB dask.array<chunksize=(1, 524288), meta=np.ndarray>
    static__train__mean             (static_feature) float64 8B dask.array<chunksize=(1,), meta=np.ndarray>
    static__train__std              (static_feature) float64 8B dask.array<chunksize=(1,), meta=np.ndarray>
Attributes:
    created_on:       2025-10-09T20:19:25
    created_with:     mllam-data-prep (https://github.com/mllam/mllam-data-prep)
    dataset_version:  v1.0.0
    mdp_version:      v0.5.0
    schema_version:   v0.6.0

## Boundary datastore

In [105]:
ds_boundary = xr.open_zarr("../data/datastore.boundary.domain03.zarr")
del ds_boundary.attrs['creation_config']
ds_boundary = ds_boundary.sel(time=forecast_timeframe)

In [106]:
ds_boundary

<xarray.Dataset> Size: 140MB
Dimensions:                         (grid_index: 99797, forcing_feature: 9,
                                     time: 19, split_name: 3, split_part: 2,
                                     static_feature: 1)
Coordinates: (12/14)
    clat                            (grid_index) float64 798kB dask.array<chunksize=(99797,), meta=np.ndarray>
    clon                            (grid_index) float64 798kB dask.array<chunksize=(99797,), meta=np.ndarray>
  * forcing_feature                 (forcing_feature) <U15 540B 't_2m' ... 't...
    forcing_feature_long_name       (forcing_feature) <U48 2kB dask.array<chunksize=(9,), meta=np.ndarray>
    forcing_feature_source_dataset  (forcing_feature) <U11 396B dask.array<chunksize=(9,), meta=np.ndarray>
    forcing_feature_units           (forcing_feature) <U8 288B dask.array<chunksize=(9,), meta=np.ndarray>
    ...                              ...
  * split_part                      (split_part) <U5 40B 'start' 'end'
  * static_feature                  (static_feature) <U3 12B 'lsm'
    static_feature_long_name        (static_feature) <U1 4B dask.array<chunksize=(1,), meta=np.ndarray>
    static_feature_source_dataset   (static_feature) <U15 60B dask.array<chunksize=(1,), meta=np.ndarray>
    static_feature_units            (static_feature) <U1 4B dask.array<chunksize=(1,), meta=np.ndarray>
  * time                            (time) datetime64[ns] 152B 2020-02-12T01:...
Data variables:
    forcing                         (forcing_feature, time, grid_index) float64 137MB dask.array<chunksize=(9, 1, 99797), meta=np.ndarray>
    forcing__train__diff_mean       (forcing_feature) float64 72B dask.array<chunksize=(9,), meta=np.ndarray>
    forcing__train__diff_std        (forcing_feature) float64 72B dask.array<chunksize=(9,), meta=np.ndarray>
    forcing__train__mean            (forcing_feature) float64 72B dask.array<chunksize=(9,), meta=np.ndarray>
    forcing__train__std             (forcing_feature) float64 72B dask.array<chunksize=(9,), meta=np.ndarray>
    splits                          (split_name, split_part) <U16 384B dask.array<chunksize=(3, 2), meta=np.ndarray>
    static                          (static_feature, grid_index) float64 798kB dask.array<chunksize=(1, 99797), meta=np.ndarray>
    static__train__mean             (static_feature) float64 8B dask.array<chunksize=(1,), meta=np.ndarray>
    static__train__std              (static_feature) float64 8B dask.array<chunksize=(1,), meta=np.ndarray>
Attributes:
    created_on:       2025-10-11T12:02:07
    created_with:     mllam-data-prep (https://github.com/mllam/mllam-data-prep)
    dataset_version:  v1.0.0
    mdp_version:      v0.5.0
    schema_version:   v0.6.0

In [ ]:
# Load inference data from neighboring domains and merge into boundary conditions